# Agentic WANDS with virtual file system + LLM judge

This notebook mirrors `configs/cheat-at-search/agentic_wands_fs_tools_judge.yml`.

We build a **virtual file system** over the WANDS catalog and then give the agent three tools: `ls`, `grep`, and `cat`. We also add **validators** and a **stopper** to guide the agent.

ELI5: imagine the product catalog is a folder of files. The agent can only list folders, search for words, and open files. We also have a teacher (LLM judge) who gives emoji feedback if the results look bad.

In [8]:
!pip install git+https://github.com/softwaredoug/cheat-at-search.git@ee2526eb8bfac087dc3f90522cc7191032e47dfd
from cheat_at_search.data_dir import mount
try:
    mount(use_gdrive=True)
except ImportError:
    from pathlib import Path
    manual_path = str(Path.home() / ".search-experiments" / "cheat-at-search")
    mount(use_gdrive=False, manual_path=manual_path)

  Cloning https://github.com/softwaredoug/cheat-at-search.git (to revision ee2526eb8bfac087dc3f90522cc7191032e47dfd) to /private/var/folders/ww/t2bpzntd1990wczd0b7px2640000gn/T/pip-req-build-jc_1c8gb
  Running command git clone --filter=blob:none --quiet https://github.com/softwaredoug/cheat-at-search.git /private/var/folders/ww/t2bpzntd1990wczd0b7px2640000gn/T/pip-req-build-jc_1c8gb
  Running command git rev-parse -q --verify 'sha^ee2526eb8bfac087dc3f90522cc7191032e47dfd'
  Running command git fetch -q https://github.com/softwaredoug/cheat-at-search.git ee2526eb8bfac087dc3f90522cc7191032e47dfd
  Resolved https://github.com/softwaredoug/cheat-at-search.git to commit ee2526eb8bfac087dc3f90522cc7191032e47dfd
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
2026-05-25 21:03:52,877 - cheat_at_s

ERROR:cheat_at_search.data_dir:Google Colab drive module not found. Ensure you're running this in Google Colab.


## Get an OpenAI Key + load corpus

This will prompt you for an OpenAI Key to interact with GPT-5.

In [9]:
import logging
import numpy as np
import pandas as pd

from openai import OpenAI
from cheat_at_search.data_dir import key_for_provider
from cheat_at_search.wands_data import corpus, judgments

OPENAI_KEY = key_for_provider("openai")
openai = OpenAI(api_key=OPENAI_KEY)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("agentic_fs_judge")

corpus = corpus.reset_index(drop=True)
doc_id_lookup = corpus["doc_id"].to_numpy()
doc_id_to_index = {doc_id: idx for idx, doc_id in enumerate(doc_id_lookup)}

corpus[["doc_id", "title", "description"]].head(3)

,doc_id,title,description
0,0,solid wood platform bed,"good , deep sleep can be quite difficult to ha..."
1,1,all-clad 7 qt . slow cooker,"create delicious slow-cooked meals , from tend..."
2,2,all-clad electrics 6.5 qt . slow cooker,prepare home-cooked meals on any schedule with...


## Sample queries (8-16)

We use a small subset to keep the notebook fast.

In [10]:
QUERY_COUNT = 12
queries = judgments[["query", "query_id"]].drop_duplicates()
queries = queries.sample(n=QUERY_COUNT, random_state=7).reset_index(drop=True)
queries

,query,query_id
0,body pillow case,118
1,deer coat hooks,455
2,tye dye duvet cover,459
3,dining table vinyl cloth,384
4,ligth bulb,305
5,garage sports storage rack,469
6,zodiac pillow,120
7,podium with locking cabinet,141
8,shoe closet,435
9,outdoor privacy wall,13


## Build virtual file system paths and contents

We simulate a file system by creating two columns:
- `path`: where the product would live as a file
- `contents`: the text inside that file (title + description)

ELI5: every product becomes a pretend text file in a folder.

In [11]:
import hashlib
import re

def _slugify_series(values: pd.Series) -> pd.Series:
    return (
        values.str.lower()
        .str.replace(r"[^a-z0-9]+", "-", regex=True)
        .str.replace(r"-+", "-", regex=True)
        .str.strip("-")
    )

def _build_filename(title_slug: str, id_slug: str, *, max_len: int = 200) -> str:
    base = f"{title_slug}-{id_slug}.txt"
    if len(base) <= max_len:
        return base
    digest = hashlib.sha1(base.encode("utf-8")).hexdigest()[:8]
    suffix = f"-{digest}-{id_slug}.txt"
    max_title_len = max_len - len(suffix)
    if max_title_len < 1:
        truncated_title = title_slug[:1] if title_slug else "d"
        return f"{truncated_title}{suffix}"
    truncated_title = title_slug[:max_title_len]
    return f"{truncated_title}{suffix}"

def _wands_path_builder(title_series: pd.Series, doc_id_series: pd.Series, corpus_df: pd.DataFrame) -> pd.Series:
    title_slug = _slugify_series(title_series).mask(lambda s: s == "", "document")
    id_slug = _slugify_series(doc_id_series).mask(lambda s: s == "", doc_id_series)
    base_name = pd.Series(
        [
            _build_filename(title, doc_id)
            for title, doc_id in zip(title_slug.tolist(), id_slug.tolist())
        ],
        index=title_series.index,
    )

    category_series = corpus_df.get("category")
    if category_series is None:
        category_series = pd.Series("", index=corpus_df.index)
    category_series = category_series.fillna("").astype(str)
    category_slug = _slugify_series(category_series)

    subcategory_series = corpus_df.get("subcategory")
    if subcategory_series is None:
        subcategory_series = pd.Series("", index=corpus_df.index)
    subcategory_series = subcategory_series.fillna("").astype(str)
    subcategory_slug = _slugify_series(subcategory_series)

    path = "/" + base_name
    has_category = category_slug != ""
    has_subcategory = subcategory_slug != ""
    path = path.where(~has_category, "/" + category_slug + "/" + base_name)
    path = path.where(
        ~(has_category & has_subcategory),
        "/" + category_slug + "/" + subcategory_slug + "/" + base_name,
    )
    return path

def ensure_wands_paths(corpus_df: pd.DataFrame) -> pd.DataFrame:
    corpus_df = corpus_df.copy()
    if "path" in corpus_df.columns or "contents" in corpus_df.columns:
        raise ValueError("Corpus already has path/contents columns.")
    title_series = corpus_df.get("title")
    if title_series is None:
        title_series = pd.Series("", index=corpus_df.index)
    title_series = title_series.fillna("").astype(str)

    description_series = corpus_df.get("description")
    if description_series is None:
        description_series = pd.Series("", index=corpus_df.index)
    description_series = description_series.fillna("").astype(str)

    doc_id_series = corpus_df.get("doc_id")
    if doc_id_series is None:
        doc_id_series = pd.Series(corpus_df.index, index=corpus_df.index)
    doc_id_series = doc_id_series.fillna("").astype(str)

    path_series = _wands_path_builder(title_series, doc_id_series, corpus_df)
    corpus_df["path"] = path_series
    corpus_df["contents"] = (
        title_series
        + " (ID: "
        + doc_id_series
        + ")\n\n"
        + description_series
    )
    return corpus_df

corpus_fs = ensure_wands_paths(corpus)
corpus_fs[["path", "contents"]].head(2)

,path,contents
0,/furniture/solid-wood-platform-bed-0.txt,"solid wood platform bed (ID: 0)\n\ngood , deep..."
1,/kitchen-tabletop/all-clad-7-qt-slow-cooker-1.txt,all-clad 7 qt . slow cooker (ID: 1)\n\ncreate ...


## Filesystem tool factory

We create three tools that simulate a file system.
- `ls_wands` lists folders and files
- `grep_wands` searches inside files
- `cat_wands` opens a file

ELI5: we are giving the agent a tiny terminal.

In [12]:
from pathlib import PurePosixPath

FS_STATE = {}

def _normalize_dir(path: str) -> str:
    if not isinstance(path, str):
        raise ValueError("path must be a string")
    trimmed = path.strip() or "/"
    if not trimmed.startswith("/"):
        trimmed = f"/{trimmed}"
    if trimmed != "/" and not trimmed.endswith("/"):
        trimmed = f"{trimmed}/"
    return trimmed

def _normalize_glob(prefix: str, glob: str) -> str:
    if not isinstance(glob, str):
        raise ValueError("glob must be a string")
    if not glob:
        glob = "*"
    if glob.startswith("/"):
        return glob
    return f"{prefix}{glob}"

def _normalize_match_pattern(pattern: str) -> str:
    normalized = pattern.strip()
    if normalized.startswith("/"):
        normalized = normalized[1:]
    return normalized

def _normalize_path(path: str) -> str:
    trimmed = path.strip().strip("\"").strip("'")
    if not trimmed:
        return ""
    if trimmed.startswith("./"):
        trimmed = trimmed[2:]
    if not trimmed.startswith("/"):
        trimmed = f"/{trimmed}"
    while "//" in trimmed:
        trimmed = trimmed.replace("//", "/")
    return trimmed

def _match_glob(pattern: str, path: str) -> bool:
    normalized_pattern = _normalize_match_pattern(pattern)
    normalized_path = path[1:] if path.startswith("/") else path
    path_obj = PurePosixPath(normalized_path)
    if path_obj.match(normalized_pattern):
        return True
    if normalized_pattern.startswith("**/") and "/" not in normalized_path:
        return path_obj.match(normalized_pattern[3:])
    return False

def _snippet_from_match(text: str, match: re.Match, window: int = 60) -> str:
    start = max(0, match.start() - window)
    end = min(len(text), match.end() + window)
    snippet = text[start:end]
    snippet = " ".join(snippet.splitlines()).strip()
    return snippet

def make_wands_fs_tools(corpus_df: pd.DataFrame):
    FS_STATE["path_series"] = corpus_df["path"].astype(str)
    FS_STATE["contents_series"] = corpus_df["contents"].astype(str)
    FS_STATE["paths"] = FS_STATE["path_series"].tolist()
    FS_STATE["path_contents"] = list(zip(FS_STATE["paths"], FS_STATE["contents_series"].tolist()))

    def ls_wands(path: str, glob: str, max_results: int = 50, agent_state=None) -> list[str] | str:
        """List files in a directory matching the glob, at most 50 results. Returns a list of paths.

        WANDS filesystem layout uses <category>/<subcategory>/<product-name-slug>-<doc-id>.txt. Example: /Furniture/Armchairs/sancroft-armchair-1234.txt. File contents are:

        <Title> (ID: <ID>)

        <Description>

        Example file contents:

        Sancroft Armchair (ID: 1234)

        A compact armchair with tailored upholstery, a supportive back, and gently flared arms designed for small spaces.
        """
        limit = max_results
        if max_results > 50:
            limit = 50
        if max_results <= 0:
            return []
        if not isinstance(path, str) or not path.strip():
            return "Error! path must be a non-empty string."
        prefix = _normalize_dir(path)
        if glob in {"*", "*/"}:
            child_map = {}
            for item in FS_STATE["paths"]:
                if not item.startswith(prefix):
                    continue
                rest = item[len(prefix):]
                if not rest:
                    continue
                child = rest.split("/", 1)[0]
                child_path = f"{prefix}{child}" if prefix != "/" else f"/{child}"
                is_dir = "/" in rest
                if child_path in child_map:
                    child_map[child_path] = child_map[child_path] or is_dir
                else:
                    child_map[child_path] = is_dir
            dir_children = sorted([path for path, is_dir in child_map.items() if is_dir])
            file_children = sorted([path for path, is_dir in child_map.items() if not is_dir])
            ordered = dir_children + file_children
            if len(ordered) > limit:
                extra = len(ordered) - limit
                ordered = ordered[:limit]
                ordered.append(f"Truncated ({extra} more)")
            return ordered
        pattern = _normalize_glob(prefix, glob)
        matches = []
        extra = 0
        for item in FS_STATE["paths"]:
            if not item.startswith(prefix):
                continue
            if _match_glob(pattern, item):
                if len(matches) < limit:
                    matches.append(item)
                else:
                    extra += 1
        matches = sorted(matches)
        if extra:
            matches.append(f"Truncated ({extra} more)")
        return matches

    def grep_wands(pattern: str, glob: str, num_results: int = 50, agent_state=None) -> list[dict[str, str]] | str:
        """Search for a regex pattern in files matching the glob, at most 50 results.

        WANDS filesystem layout uses <category>/<subcategory>/<product-name-slug>-<doc-id>.txt. Example: /Furniture/Armchairs/sancroft-armchair-1234.txt. File contents are:

        <Title> (ID: <ID>)

        <Description>

        Example file contents:

        Sancroft Armchair (ID: 1234)

        A compact armchair with tailored upholstery, a supportive back, and gently flared arms designed for small spaces.
        """
        limit = num_results
        if num_results > 50:
            limit = 50
        if num_results <= 0:
            return []
        try:
            regex = re.compile(pattern)
        except re.error:
            return f"Error! Invalid regex pattern: {pattern}"
        match_pattern = _normalize_glob("/", glob)
        results = []
        extra = 0
        for path, contents in FS_STATE["path_contents"]:
            if not _match_glob(match_pattern, path):
                continue
            match = regex.search(contents)
            if not match:
                continue
            if len(results) < limit:
                results.append({"path": path, "snippet": _snippet_from_match(contents, match)})
            else:
                extra += 1
        if extra:
            results.append({"path": "", "snippet": f"Truncated ({extra} more)"})
        return results

    def cat_wands(path: str, agent_state=None) -> str:
        """Return the contents of a file as a string.

        WANDS filesystem layout uses <category>/<subcategory>/<product-name-slug>-<doc-id>.txt. Example: /Furniture/Armchairs/sancroft-armchair-1234.txt. File contents are:

        <Title> (ID: <ID>)

        <Description>

        Example file contents:

        Sancroft Armchair (ID: 1234)

        A compact armchair with tailored upholstery, a supportive back, and gently flared arms designed for small spaces.
        """
        if not isinstance(path, str) or not path.strip():
            return "Error! path must be a non-empty string."
        normalized_path = _normalize_path(path)
        if not normalized_path:
            return "Error! path must be a non-empty string."
        matches = FS_STATE["contents_series"][FS_STATE["path_series"] == normalized_path]
        if matches.empty and normalized_path != path:
            matches = FS_STATE["contents_series"][FS_STATE["path_series"] == path]
        if matches.empty:
            filename = PurePosixPath(normalized_path or path).name
            if filename:
                filename_matches = FS_STATE["contents_series"][FS_STATE["path_series"].str.endswith(f"/{filename}")]
                if len(filename_matches) == 1:
                    return str(filename_matches.iloc[0])
                if len(filename_matches) > 1:
                    return f"Error! Multiple files found for filename: {filename}"
            return f"Error! No file found for path: {path}"
        if len(matches) > 1:
            return f"Error! Multiple files found for path: {path}"
        return str(matches.iloc[0])

    return [ls_wands, grep_wands, cat_wands]

tools = make_wands_fs_tools(corpus_fs)
ls_tool, grep_tool, cat_tool = tools
ls_tool("/", "*", max_results=5)

['/accommodations',
 '/appliances',
 '/baby-kids',
 '/bath-rugs-mats',
 '/bed-accessories',
 'Truncated (1606 more)']

## Validators and stoppers (ELI5)

Validators are like rules the agent must satisfy **before** it can stop. If a validator fails, we add its message and try again.

Stoppers are the opposite: if a stopper condition is met, we stop looping. If not, we add its message and try again.

We define them as free functions so you can read and modify them easily.

In [15]:
from typing import Literal
from pydantic import BaseModel, Field
from cheat_at_search.agent.openai_agent import OpenAIAgent

AllowedEmoji = Literal["😃", "😐", "😞"]

class GradedSearchResult(BaseModel):
    emoji: AllowedEmoji = Field(description="Emoji relevance label for this result.")
    title: str = Field(description="Document title for the judged result.")
    doc_id: str = Field(description="Document ID for the judged result.")

class LLMJudgeResponse(BaseModel):
    graded_results: list[GradedSearchResult] = Field(
        default_factory=list,
        description="Ordered list of graded search results with emoji labels.",
    )

def validate_min_results(resp, min_results=10):
    results = getattr(resp, "ranked_results", None) if resp else None
    count = len(results or [])
    if count >= min_results:
        return True
    return "Please return at least 10 results to give the user a good variety to choose from."

def _render_results_for_judge(ranked_doc_ids):
    lines = []
    for idx, doc_id in enumerate(ranked_doc_ids, start=1):
        try:
            doc_id_int = int(doc_id)
        except (TypeError, ValueError):
            continue
        row = None
        if doc_id_int in doc_id_to_index:
            row = corpus.iloc[doc_id_to_index[doc_id_int]]
        title = str(row.get("title", "")) if row is not None else ""
        description = str(row.get("description", "")) if row is not None else ""
        if len(description) > 200:
            description = description[:197] + "..."
        lines.append(f"{idx}. {title} (ID: {doc_id_int})\n{description}")
    return "\n\n".join(lines)

def validate_llm_judge(query, resp, model="gpt-5-mini", reasoning="medium"):
    ranked = getattr(resp, "ranked_results", None) if resp else None
    ranked_doc_ids = list(ranked or [])
    judge_prompt = (
        "You are a helpful assistant that judges the relevance of search results to a query.\n\n"
        "Query: {query}\n\n"
        "Results:\n{results}\n\n"
        "Please rate the relevance of these results to the query using emojis of how well they satisfy the query.\n\n"
        "Respond as a list of graded results with fields: emoji, title, doc_id."
    )
    results_block = _render_results_for_judge(ranked_doc_ids)
    inputs = [{"role": "user", "content": judge_prompt.format(query=query, results=results_block)}]
    judge = OpenAIAgent(
        tools=[],
        model=f"openai/{model}" if "/" not in model else model,
        response_model=LLMJudgeResponse,
        reasoning_level=reasoning,
    )
    judge_resp, _, _ = judge.chat(inputs=inputs, agent_state=None, logger=logger)
    graded = judge_resp.output_parsed.graded_results or []
    if not graded:
        return "Please return more relevant results to better help the user find what they're looking for."
    if any(item.emoji in {"😞", "😐"} for item in graded):
        eval_block = "\n".join(
            f"{idx}. {item.emoji} {item.title} (ID: {item.doc_id})"
            for idx, item in enumerate(graded, start=1)
        )
        return (
            "Please return more relevant results to better help the user find what they're looking for.\n\n"
            "LLM evaluations:\n\n" + eval_block
        )
    return True

def stop_after_tool_calls(num_tool_calls, min_calls=16):
    if num_tool_calls >= min_calls:
        return True
    return "Please make at least 16 tool calls to gather enough information before returning results."

## Agentic strategy with validators + stopper

We wrap the validators and stopper into a simple loop around `OpenAIAgent`.

ELI5: we keep asking the agent until it satisfies our rules or hits the stop condition.

In [16]:
from pydantic import BaseModel, Field
from cheat_at_search.agent.openai_agent import OpenAIAgent
from cheat_at_search.strategy import SearchStrategy

SYSTEM_PROMPT = """
You take user search queries and use filesystem tools to find relevant products.

Use ls_wands to explore directories, grep_wands to search within files, and cat_wands
to inspect specific product files.

It's important to return results ranked from most to least relevant based on the user query.

Gather results until you have 10 best matches you can find. It's important to return at least 10.
Return the *DOC IDs*, not paths, in the response.
"""

class SearchResults(BaseModel):
    ranked_results: list[str] = Field(description="Top ranked search results (their doc_ids) when complete")

def _count_tool_calls(inputs):
    return sum(1 for item in inputs if isinstance(item, dict) and item.get("type") == "function_call_output")

class AgenticFilesystemJudgeStrategy(SearchStrategy):
    def __init__(self, corpus_df: pd.DataFrame, tools: list[callable], model: str = "gpt-5-mini", workers: int = 1):
        self.tools = tools
        self.model = model
        super().__init__(corpus_df, workers=workers)

    def search(self, query: str, k: int = 10):
        inputs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"The user's query: {query}"},
        ]
        agent_state = {}
        agent = OpenAIAgent(
            tools=self.tools,
            model=f"openai/{self.model}" if "/" not in self.model else self.model,
            response_model=SearchResults,
            reasoning_level="medium",
        )
        max_loops = 10
        for _ in range(max_loops):
            resp = agent.loop(inputs=inputs, agent_state=agent_state, logger=logger)
            tool_calls = _count_tool_calls(inputs)

            # Validators (first failing wins)
            msg = validate_min_results(resp, min_results=10)
            if msg is not True:
                inputs.append({"role": "user", "content": msg})
                continue
            msg = validate_llm_judge(query, resp)
            if msg is not True:
                inputs.append({"role": "user", "content": msg})
                continue

            # Stopper
            msg = stop_after_tool_calls(tool_calls, min_calls=16)
            if msg is True:
                break
            inputs.append({"role": "user", "content": msg})

        ranked = [doc_id for doc_id in (resp.ranked_results or [])]
        ranked = [doc_id_to_index.get(int(doc_id), -1) for doc_id in ranked if str(doc_id).isdigit()]
        ranked = [idx for idx in ranked if idx >= 0][:k]
        return ranked, [1.0] * len(ranked)

strategy = AgenticFilesystemJudgeStrategy(corpus_fs, tools, workers=1)
strategy.search(queries.loc[0, "query"], k=5)

([10980, 1981, 14956, 20960, 20963], [1.0, 1.0, 1.0, 1.0, 1.0])

## Run the small benchmark

We run `run_strategy` on a small subset of queries and compute mean NDCG.

ELI5: NDCG is a score from 0 to 1 that says how good the ranking is (higher is better).

In [17]:
from cheat_at_search.search import run_strategy, ndcgs

results = run_strategy(strategy, judgments, num_queries=QUERY_COUNT, seed=7, cache=False)
ndcg_series = ndcgs(results)
float(ndcg_series.mean())

Searching: 100%|███████████████████████████████████████████████████████████████████████| 12/12 [57:43<00:00, 288.60s/it]


0.6864471842117147

In [ ]:
ndcg_series